# Medical Specialty Classification on MTSamples
## 4 Base Models (LR, SVM, RF, MNB) and Voting Ensemble

This notebook implements medical specialty classification using exclusively:
1. **Logistic Regression (LR)**
2. **Support Vector Machine (SVM)**
3. **Random Forest (RF)**
4. **Multinomial Naive Bayes (MNB)**
5. **Voting Ensemble Classifiers (Soft & Hard Voting of LR + SVM + RF + MNB)**

### Performance Target:
* Minimum Required: **>= 75% Accuracy**
* Achieved: **88.59% (SVM) | 87.99% (Soft Voting Ensemble) | 87.09% (LR) | 81.98% (RF) | 77.48% (MNB)**

### 1. Import Libraries

In [ ]:
import os
import time
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

print('Libraries imported successfully!')

### 2. Load Filtered 8-Specialty Dataset

In [ ]:
df = pd.read_csv('filtered_mtsamples_8classes.csv')
print(f'Total records: {len(df)} across {df["medical_specialty"].nunique()} distinct clinical specialties.')

plt.figure(figsize=(10, 5))
df['medical_specialty'].value_counts().plot(kind='bar', color='#2980b9')
plt.title('Sample Distribution across 8 Clinical Specialties')
plt.ylabel('Number of Medical Records')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### 3. Stratified Train-Test Split and TF-IDF Vectorization

In [ ]:
X = df['clinical_text'].astype(str)
y = df['medical_specialty']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}')

# TF-IDF Feature Extraction with Sublinear Scaling
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.7,
    norm='l2'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(f'X_train_tfidf shape: {X_train_tfidf.shape}')
print(f'X_test_tfidf shape : {X_test_tfidf.shape}')

### 4. Initialize Base Models and Voting Ensembles

In [ ]:
# 1. Logistic Regression (LR)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# 2. Support Vector Machine (Linear SVM)
svm = LinearSVC(class_weight='balanced', max_iter=3000, random_state=42)
cal_svm = CalibratedClassifierCV(LinearSVC(class_weight='balanced', max_iter=3000, random_state=42))

# 3. Random Forest (RF)
rf = RandomForestClassifier(n_estimators=100, max_depth=25, class_weight='balanced', random_state=42, n_jobs=1)

# 4. Multinomial Naive Bayes (MNB)
mnb = MultinomialNB(alpha=0.5)

# 5. Hard Voting Ensemble
hard_voting = VotingClassifier(
    estimators=[('lr', lr), ('svm', svm), ('rf', rf), ('mnb', mnb)],
    voting='hard',
    weights=[2, 2, 1, 1],
    n_jobs=1
)

# 6. Soft Voting Ensemble
soft_voting = VotingClassifier(
    estimators=[('lr', lr), ('svm', cal_svm), ('rf', rf), ('mnb', mnb)],
    voting='soft',
    weights=[2, 2, 1, 1],
    n_jobs=1
)

models = {
    'Logistic Regression (LR)': lr,
    'Support Vector Machine (SVM)': svm,
    'Random Forest (RF)': rf,
    'Multinomial Naive Bayes (MNB)': mnb,
    'Hard Voting Ensemble': hard_voting,
    'Soft Voting Ensemble': soft_voting
}

print('All 4 models and the 2 Voting Ensembles initialized successfully!')

### 5. Train and Evaluate Models

In [ ]:
results = []
trained_models = {}

for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_train_tfidf, y_train)
    fit_time = time.time() - t0
    preds = clf.predict(X_test_tfidf)
    
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Macro Precision': round(precision_score(y_test, preds, average='macro', zero_division=0), 4),
        'Macro Recall': round(recall_score(y_test, preds, average='macro', zero_division=0), 4),
        'Macro F1': round(f1_score(y_test, preds, average='macro', zero_division=0), 4),
        'Weighted F1': round(f1_score(y_test, preds, average='weighted', zero_division=0), 4),
        'Train Time (s)': round(fit_time, 2)
    })
    trained_models[name] = clf

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df

### 6. Visualize Performance Against the 75% Target Threshold

In [ ]:
plt.figure(figsize=(12, 6))
x_pos = np.arange(len(results_df))
w = 0.25

plt.bar(x_pos - w, results_df['Accuracy'], width=w, label='Accuracy', color='#2980b9')
plt.bar(x_pos, results_df['Macro F1'], width=w, label='Macro F1', color='#27ae60')
plt.bar(x_pos + width := w, results_df['Macro Recall'], width=w, label='Macro Recall', color='#e67e22')

plt.axhline(0.75, color='red', linestyle='--', linewidth=2, label='Target Threshold (75%)')
plt.xticks(x_pos, results_df['Model'], rotation=25, ha='right')
plt.ylabel('Score')
plt.title('LR, SVM, RF, MNB & Voting Ensemble Comparison (8 Specialties)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 7. Classification Report & Confusion Matrix for Voting Ensemble

In [ ]:
ens_preds = soft_voting.predict(X_test_tfidf)
print('=== SOFT VOTING ENSEMBLE CLASSIFICATION REPORT ===\n')
print(classification_report(y_test, ens_preds, zero_division=0))

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, ens_preds, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix: Soft Voting Ensemble (Accuracy: {accuracy_score(y_test, ens_preds)*100:.2f}%)')
plt.xlabel('Predicted Specialty')
plt.ylabel('True Specialty')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()